In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Phase 6").getOrCreate()

# Dirty Customers Dataset

In [0]:
customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

# Dirty Orders Dataset

In [0]:
orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])

In [0]:
# Convert date column
orders = orders.withColumn("order_date", to_date(col("order_date")))

In [0]:
null_values = customers.join(orders, "customer_id", "left")
rows_with_null_values = null_values.filter(col('customer_id').isNull() | col('name').isNull() | col('email').isNull() | col('amount').isNull() | col('order_id').isNull() | col('order_date').isNull() | col('city').isNull())
rows_with_null_values.display()

customer_id,name,email,city,order_id,order_date,amount
1,John Doe,john@example.com,Hyderabad,105,2024-01-05,null
3,null,bob@example.com,Bangalore,103,2024-01-03,-500
4,David,null,Mumbai,null,null,null
6,Frank,frank@example.com,Delhi,null,null,null


In [0]:
# fill null values in customers's name and email as unknown, amount as 0 in orders
customers = customers.fillna("unkown", subset = ["name","email"])
orders = orders.fillna(0,subset = ["amount"])

In [0]:
customers.display()
orders.display()

customer_id,name,email,city
1,John Doe,john@example.com,Hyderabad
2,Alice,alice@example.com,Chennai
3,unkown,bob@example.com,Bangalore
4,David,unkown,Mumbai
5,Eva,eva@example.com,Hyderabad
6,Frank,frank@example.com,Delhi


order_id,customer_id,order_date,amount
101,1,2024-01-01,1000
102,2,2024-01-02,2000
103,3,2024-01-03,-500
104,99,2024-01-04,1500
105,1,2024-01-05,0
106,5,2024-01-06,3000
107,5,2024-01-07,3000


In [0]:
# valid dataset (inner join)
valid_dataset = customers.join(orders, "customer_id", "inner")
valid_dataset.display()

customer_id,name,email,city,order_id,order_date,amount
1,John Doe,john@example.com,Hyderabad,105,2024-01-05,0
1,John Doe,john@example.com,Hyderabad,101,2024-01-01,1000
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000
3,unkown,bob@example.com,Bangalore,103,2024-01-03,-500
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000


In [0]:
valid_dataset = valid_dataset.filter(col("amount") >= 0)
valid_dataset.display()

customer_id,name,email,city,order_id,order_date,amount
1,John Doe,john@example.com,Hyderabad,105,2024-01-05,0
1,John Doe,john@example.com,Hyderabad,101,2024-01-01,1000
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000


In [0]:
# invalid foriegn keys
invalid_fk = orders.join(customers,"customer_id","left_anti")
invalid_fk.display()

customer_id,order_id,order_date,amount
99,104,2024-01-04,1500


In [0]:
# compare rows count accross joining
print("valid_dataset count", valid_dataset.count())
print("rows_with_null_values count", rows_with_null_values.count())
print("invalid_fk count", invalid_fk.count())

valid_dataset count 5
rows_with_null_values count 4
invalid_fk count 1


In [0]:
# top 3 customers per city with highest total order amount
top_customers = valid_dataset.groupBy("city").agg(sum("amount").alias("total_amount")).orderBy(col("total_amount").desc()).limit(3)
top_customers.display()

city,total_amount
Hyderabad,7000
Chennai,2000


In [0]:
# top 3 customers per each city with highest total order amount using window function
from pyspark.sql.window import Window

customer_total = valid_dataset.groupBy("customer_id","name","city").agg(sum("amount").alias("total_amount"))
window_city = Window.partitionBy("city").orderBy(col("total_amount").desc())
top_3_each_city = customer_total.withColumn("rank", rank().over(window_city)).filter(col("rank") <= 3)
top_3_each_city.display()

customer_id,name,city,total_amount,rank
2,Alice,Chennai,2000,1
5,Eva,Hyderabad,6000,1
1,John Doe,Hyderabad,1000,2


# Running Total of Sales (per customer)

In [0]:
window_spec = Window.partitionBy("customer_id") \
    .orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_df = valid_dataset.withColumn(
    "running_total",
    sum("amount").over(window_spec)
)

running_total_df.display()

customer_id,name,email,city,order_id,order_date,amount,running_total
1,John Doe,john@example.com,Hyderabad,101,2024-01-01,1000,1000
1,John Doe,john@example.com,Hyderabad,105,2024-01-05,0,1000
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000,2000
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000,3000
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000,6000


In [0]:
# Aggregate total spend
customer_totals = valid_dataset.groupBy("customer_id", "name") \
    .agg(sum("amount").alias("total_spend"))

# Rank globally
window_spec = Window.orderBy(col("total_spend").desc())

ranked_customers = customer_totals.withColumn(
    "rank",
    rank().over(window_spec)
)

ranked_customers.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,name,total_spend,rank
5,Eva,6000,1
2,Alice,2000,2
1,John Doe,1000,3


In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

lag_df = valid_dataset.withColumn(
    "previous_amount",
    lag("amount", 1).over(window_spec)
).withColumn(
    "amount_diff",
    col("amount") - col("previous_amount")
)

lag_df.display()

customer_id,name,email,city,order_id,order_date,amount,previous_amount,amount_diff
1,John Doe,john@example.com,Hyderabad,101,2024-01-01,1000,null,null
1,John Doe,john@example.com,Hyderabad,105,2024-01-05,0,1000,-1000
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000,null,null
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000,null,null
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000,3000,0


## Date Analysis- Extract month from date- Monthly sales aggregation- Calculate difference between dates- Trend analysis by month

In [0]:
df = valid_dataset.withColumn(
    "year_month",
    date_format("order_date", "yyyy-MM")
)
df.display()
monthly_sales = df.groupBy("year_month") \
    .agg(sum("amount").alias("total_sales"))

window_spec = Window.orderBy("year_month")

final_df = monthly_sales \
    .withColumn("prev_sales", lag("total_sales").over(window_spec)) \
    .withColumn("growth", col("total_sales") - col("prev_sales"))

final_df.display()

customer_id,name,email,city,order_id,order_date,amount,year_month
1,John Doe,john@example.com,Hyderabad,105,2024-01-05,0,2024-01
1,John Doe,john@example.com,Hyderabad,101,2024-01-01,1000,2024-01
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000,2024-01
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000,2024-01
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000,2024-01


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year_month,total_sales,prev_sales,growth
2024-01,9000,null,null
